
# 03 - Non-neural baselines

This notebook evaluates four pre-registered configurations on the same rolling-origin folds: zero, weekly seasonal naive, a four-week seasonal average, and one global LightGBM model. It writes `metrics/baseline_results.csv` inside the reported run directory, OOF predictions, figures, provenance metadata, and optional local execution records. No recurrent model is trained.

**Execution records.** New outputs are written under `artifacts/<mode>/<run_id>/`.
Previous numerical interpretations describe the original study, not a new execution;
they do not describe a new verified execution. Run cells in order, then retain the
reported run ID and completion manifest. Rerunning writers after completion requires a new run.
`RETAIL_FORECAST_POLICY` selects `competition_v1` (historical Kaggle-supplied future covariates)
or `asof_v1` (default; no oil or Event rows). Outputs are regenerated on execution.


## Ejecución de la entrega

Ejecuta todas las celdas en orden en una sesión limpia. El modo predeterminado
`full` utiliza datos reales; el notebook 04 requiere GPU. Para comprobar solo
esquema y ejecución, define `os.environ["RETAIL_FORECAST_MODE"] = "smoke"` antes
de la preparación. Los resultados sintéticos no miden precisión.

Mantén los siete CSV en `data/raw` del espacio de trabajo o usa el ZIP de Kaggle.
No escribas credenciales en celdas ni salidas. Conserva el notebook ejecutado y
el directorio de resultados antes de desconectar Colab.

In [ ]:
# Immutable package setup; see docs/01_reproducibility.md for clean local mode.
import importlib.util
import os
import re
import runpy
import subprocess
import sys
import tempfile
from getpass import getpass
from pathlib import Path

# The published academic URL can be supplied through the environment.
REPO_URL = os.environ.get("RETAIL_FORECAST_REPO_URL", "").strip() or globals().get("REPO_URL", "")
if not REPO_URL:
    if (Path.cwd() / "pyproject.toml").is_file():
        REPO_URL = str(Path.cwd())
    else:
        REPO_URL = "https://github.com/camiloandcu/Retail-Demand-Forecasting.git"
if not REPO_URL:
    raise ValueError("Indica la URL del repositorio académico publicado.")
CODE_REVISION = "4a6748b8143ef09c83d899576f98c9572d934ee1"
MODELS = True
if sys.version_info[:2] not in {(3, 10), (3, 11), (3, 12), (3, 13)}:
    raise RuntimeError("Use Python 3.10–3.13 before installing or acquiring data.")
if not re.fullmatch(r"[0-9a-f]{40}", CODE_REVISION):
    raise RuntimeError("CODE_REVISION must be a reviewed full commit SHA.")
IN_COLAB = (
    importlib.util.find_spec("google") is not None
    and importlib.util.find_spec("google.colab") is not None
)
REPO_ROOT = Path(
    os.environ.get(
        "RETAIL_FORECAST_CHECKOUT",
        str(Path("/content" if IN_COLAB else tempfile.gettempdir()) / f"retail-academic-{CODE_REVISION}"),
    )
).absolute()
if REPO_ROOT.is_symlink() or any(p.is_symlink() for p in REPO_ROOT.parents):
    raise RuntimeError("Checkout path must not contain symlinks; use a new directory.")


def checkout_git(*arguments):
    return subprocess.run(
        ["git", "-C", str(REPO_ROOT), *arguments],
        check=True,
        capture_output=True,
        text=True,
    ).stdout.strip()


if not REPO_ROOT.exists():
    subprocess.run(["git", "clone", "--no-checkout", REPO_URL, str(REPO_ROOT)], check=True)
    # A squash merge can leave the reviewed SHA reachable only by its PR ref.
    subprocess.run(
        ["git", "-C", str(REPO_ROOT), "fetch", "--no-tags", "origin", CODE_REVISION],
        check=True,
    )
    subprocess.run(
        ["git", "-C", str(REPO_ROOT), "checkout", "--detach", CODE_REVISION],
        check=True,
    )
# Validate before executing even the shared helper from this checkout.
try:
    valid = (
        checkout_git("rev-parse", "--show-toplevel") == str(REPO_ROOT)
        and checkout_git("remote", "get-url", "origin") == REPO_URL
        and checkout_git("rev-parse", "HEAD") == CODE_REVISION
        and not checkout_git("status", "--porcelain", "--untracked-files=all")
    )
except subprocess.CalledProcessError:
    valid = False
if not valid:
    raise RuntimeError(
        "Dirty, incomplete or mismatched checkout. Preserve it; use a new directory."
    )
helper = runpy.run_path(str(REPO_ROOT / "src/retail_forecast/notebook_runtime.py"))
runtime_evidence = helper["bootstrap"](
    REPO_ROOT, CODE_REVISION, REPO_URL, models=MODELS, colab=IN_COLAB
)
print(
    "Notebook revision:",
    os.getenv("RETAIL_FORECAST_NOTEBOOK_REVISION", "unrecorded; retain immutable notebook URL"),
)
WORKSPACE_ROOT = globals().get(
    "WORKSPACE_ROOT", Path.cwd() if (Path.cwd() / "pyproject.toml").is_file() else REPO_ROOT
)
os.chdir(REPO_ROOT)

In [ ]:
from dataclasses import replace

import pandas as pd
from IPython.display import Image, display

from retail_forecast.acquisition import acquire_competition_data
from retail_forecast.artifacts import (
    capture_inputs,
    finish_run,
    read_completed_run,
    run_result,
    write_run_artifact,
)
from retail_forecast.config import load_config
from retail_forecast.data import (
    required_files_exist,
    validate_dataset,
    write_synthetic_dataset,
)
from retail_forecast.eda import validate_kaggle_hashes
from retail_forecast.experiments import MODEL_IDS, run_baseline_experiments
from retail_forecast.notebook_runs import notebook_config_path, prepare_notebook_run
from retail_forecast.reproducibility import set_global_seed

MODE = os.getenv("RETAIL_FORECAST_MODE", "full").strip().lower()
# The information regime is explicit; results from different policies never pool.
POLICY = os.getenv("RETAIL_FORECAST_POLICY", "asof_v1").strip().lower()
config = load_config(notebook_config_path(MODE, POLICY))
config = replace(
    config,
    paths=replace(
        config.paths,
        raw_data_dir=WORKSPACE_ROOT / config.paths.raw_data_dir.relative_to(REPO_ROOT),
        artifacts_dir=WORKSPACE_ROOT / config.paths.artifacts_dir.relative_to(REPO_ROOT),
    ),
)
run_context, config = prepare_notebook_run(config, kind="baseline")
print("Run ID:", run_context.run_id, "| root:", run_context.root)
set_global_seed(config.runtime.seed)
print(
    f"mode={MODE} | policy={config.data.availability_policy} "
    f"| configurations={len(MODEL_IDS)} | tree={config.baseline.tree_model}"
)

## Data

Existing CSVs and a locally available competition ZIP are reused automatically. If neither is present, the next cell asks for a Kaggle API token using hidden input.

In [ ]:
if config.data.source == "synthetic":
    if not required_files_exist(config.paths.raw_data_dir):
        write_synthetic_dataset(config)
elif not required_files_exist(config.paths.raw_data_dir):
    acquire_competition_data(
        config.paths.raw_data_dir,
        (WORKSPACE_ROOT / "data", WORKSPACE_ROOT),
        token_reader=getpass,
    )
capture_inputs(run_context, input_dir=config.paths.raw_data_dir)
if MODE == "full":
    validate_kaggle_hashes(config.paths.raw_data_dir)
dataset_summary = validate_dataset(config)
display(dataset_summary.to_dict())

## Decisions fixed before training

LightGBM is the only tree model. The training code declares the six encoded categorical columns directly, learns `log1p(sales)`, and clips negative predictions to zero before RMSLE.

The comparison uses four fixed configurations. Report the recurrent model on the same temporal folds, even if it does not improve LightGBM. Synthetic results validate execution only. A historical 2% threshold remains in inherited metadata; it is not an academic acceptance requirement.

In [ ]:
run = run_baseline_experiments(
    config,
    run=run_context,
    tracking=True,
)

## Quality and cost

The tables compare global RMSLE, each validation fold, and each forecast day. The figures below show the global and horizon comparisons. Store, family, promotion, and calendar-regime scores remain in `baseline_results.csv`.

In [ ]:
display(run["cost_quality"])
display(
    run["results"]
    .loc[run["results"]["scope"].isin(["global", "fold"])]
    .pivot_table(index=["scope", "segment"], columns="model_id", values="rmsle")
)
display(
    run["results"]
    .loc[run["results"]["scope"].eq("horizon")]
    .assign(segment=lambda frame: frame["segment"].astype(int))
    .pivot(index="segment", columns="model_id", values="rmsle")
)
display(Image(filename=str(run_context.root / "figures/baselines/01_global_rmsle.png")))
display(Image(filename=str(run_context.root / "figures/baselines/02_horizon_rmsle.png")))

In [ ]:
import matplotlib.pyplot as plt

horizon_without_zero = (
    run["results"]
    .loc[run["results"]["scope"].eq("horizon") & run["results"]["model_id"].ne("zero")]
    .assign(segment=lambda frame: frame["segment"].astype(int))
    .sort_values(["model_id", "segment"])
)

labels = {
    "seasonal_naive_7d": "Seasonal naive (7d)",
    "seasonal_average_4w": "Seasonal average (4w)",
    "lightgbm_global": "LightGBM",
}
figure, axis = plt.subplots(figsize=(9, 4.5))
for model_id, model_frame in horizon_without_zero.groupby("model_id", sort=False):
    axis.plot(
        model_frame["segment"],
        model_frame["rmsle"],
        marker="o",
        label=labels[model_id],
    )
axis.set(
    xlabel="Forecast horizon",
    ylabel="OOF RMSLE",
    title="Error by forecast horizon (excluding the zero check)",
)
axis.set_xticks(sorted(horizon_without_zero["segment"].unique()))
axis.grid(axis="y", alpha=0.25)
axis.legend()
figure.tight_layout()

figure_relative = "figures/baselines/02_horizon_rmsle_without_zero.png"
write_run_artifact(
    run_context,
    figure_relative,
    lambda path: figure.savefig(path, format="png", dpi=140, bbox_inches="tight"),
    kind="figure",
)
figure_path = run_context.root / figure_relative
plt.show()
print(f"Saved: {figure_path}")

## Reading the comparison

The zero check towers above the other methods and stretches the figures' scale. Use the table to compare the useful models: LightGBM leads, while the four-week average is the strongest simple baseline. We now have both a demanding ML reference and an understandable comparator.

Every useful method is less accurate in the last fold. A good global score can hide a weak period, so the next model must be checked fold by fold. This table does not explain what caused the change.

LightGBM leads across the forecast days, but its error varies rather than rising smoothly. Lead days also fall on different calendar dates, so the curve alone cannot explain the variation. Keep the same folds and compare every forecast day in the next experiment.

This describes the original study; rerun tables above provide the current execution results. Smoke plots are synthetic and do not measure forecasting quality.

## Reference models for the recurrent comparison

The four-week average is the interpretable baseline; LightGBM is the strongest inherited reference. Compare the recurrent model on identical folds and forecast keys, then inspect errors by period and horizon. Improvement is an empirical question, not a condition for completing this assignment.

In [ ]:
display(run["manifest"])
for name, path in run["paths"].items():
    print(f"{name}: {path}")

## Where the error sits

The global RMSLE hides differences between types of forecast days and series. The next cell reads the scores already calculated in `baseline_results.csv`; it does not fit another model. These are descriptive slices of the same LightGBM OOF predictions.

In [ ]:
tree_segments = run["results"].loc[run["results"]["model_id"].eq("lightgbm_global")]
for scope in ("promotion", "temporal_regime"):
    print(f"LightGBM by {scope}")
    display(tree_segments.loc[tree_segments["scope"].eq(scope), ["segment", "rows", "rmsle"]])
for scope in ("store", "family"):
    print(f"Five highest-error {scope} segments")
    display(
        tree_segments.loc[tree_segments["scope"].eq(scope), ["segment", "rows", "rmsle"]]
        .sort_values("rmsle", ascending=False)
        .head(5)
    )
print(f"Complete segment results: {run['paths']['results']}")

### What these slices tell us

**Promotion.** Non-promoted rows have higher error. Keep both groups in the next comparison, but do not call this a promotion effect: their stores, families, dates, and sales levels may differ.

**Calendar regime.** Holiday/event rows have the highest error and form a much smaller group. Inspect those predictions before blaming the event; the other regimes are fairly close.

**Stores and families.** Error varies across the panel. The family with the most zero-sales days in the EDA is not in this highest-error list, so frequent zeros alone do not explain the ranking.

Keep these slices for the recurrent comparison. Before changing features or sampling for a group, inspect its OOF predictions to see what mistakes the model makes.

In [ ]:
checks = {
    "four configurations only": len(MODEL_IDS) == 4,
    "one tree family only": config.baseline.tree_model == "lightgbm",
    "official result artifact": run["paths"]["results"].is_file(),
    "OOF artifact": run["paths"]["oof"].is_file(),
    "tracking status recorded": run["manifest"]["tracking"]["status"]
    in {"enabled", "disabled", "failed"},
}
gate = pd.DataFrame(
    [{"check": name, "status": "PASS" if passed else "FAIL"} for name, passed in checks.items()]
)
display(gate)
if not all(checks.values()):
    raise RuntimeError("Baseline gate failed.")
print(f"Baseline execution checks: {MODE.upper()} PASS")
finish_run(run_context)
verified_run = read_completed_run(
    run_context.root,
    expected_mode=MODE,
    expected_source=config.data.source,
)
print("Verified completed run:", run_result(run_context))